In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)
from pyspark.sql.functions import (
    col, when, to_date, dayofweek, month, hour,
    count, avg, sum as _sum
)
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml import Pipeline
from pyspark.ml.classification import (
    LogisticRegression, RandomForestClassifier
)
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator, MulticlassClassificationEvaluator
)
from pyspark import StorageLevel

spark = (
    SparkSession.builder
    .appName("s04-clasificacion-congestion-joel")
    .master("local[*]")
    .config("spark.ui.port", "4042")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

ORIGEN_DATOS = "/opt/s04-ml-distribuido-regresion/data"
ARTIFACTS = "/opt/s04-ml-distribuido-regresion/artifacts"

print("✅ Spark listo")

✅ Spark listo


26/09/11 18:00:47 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## Fase 1 — Business Understanding

**Objetivo:** Predecir `congestion_level` (nivel de congestión de un aeropuerto
en una hora específica) a partir del flujo histórico de vuelos.

**Variable objetivo:** `congestion_level`
- 0 = Baja (pocos vuelos en esa hora)
- 1 = Media
- 2 = Alta (saturación)

**Alcance:** Comparar LogisticRegression vs RandomForestClassifier.
Reportar AUC, F1 y Accuracy. Guardar el ganador.

**Decisión que habilita:** Planificar turnos de personal y uso de pistas
según la congestión esperada por aeropuerto y hora.

In [2]:
schema_vuelos = StructType([
    StructField("year", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("day_of_month", IntegerType(), True),
    StructField("day_of_week", IntegerType(), True),
    StructField("fl_date", StringType(), True),
    StructField("op_unique_carrier", StringType(), True),
    StructField("op_carrier_fl_num", IntegerType(), True),
    StructField("origin", StringType(), True),
    StructField("origin_city_name", StringType(), True),
    StructField("origin_state_nm", StringType(), True),
    StructField("dest", StringType(), True),
    StructField("dest_city_name", StringType(), True),
    StructField("dest_state_nm", StringType(), True),
    StructField("crs_dep_time", StringType(), True),
    StructField("dep_time", StringType(), True),
    StructField("dep_delay", DoubleType(), True),
    StructField("taxi_out", DoubleType(), True),
    StructField("wheels_off", StringType(), True),
    StructField("wheels_on", StringType(), True),
    StructField("taxi_in", DoubleType(), True),
    StructField("crs_arr_time", StringType(), True),
    StructField("arr_time", StringType(), True),
    StructField("arr_delay", DoubleType(), True),
    StructField("cancelled", DoubleType(), True),
    StructField("cancellation_code", StringType(), True),
    StructField("diverted", DoubleType(), True),
    StructField("crs_elapsed_time", DoubleType(), True),
    StructField("actual_elapsed_time", DoubleType(), True),
    StructField("air_time", DoubleType(), True),
    StructField("distance", DoubleType(), True),
    StructField("carrier_delay", DoubleType(), True),
    StructField("weather_delay", DoubleType(), True),
    StructField("nas_delay", DoubleType(), True),
    StructField("security_delay", DoubleType(), True),
    StructField("late_aircraft_delay", DoubleType(), True),
])

df = spark.read.csv(
    f"{ORIGEN_DATOS}/flight_data_2024.csv",
    header=True,
    schema=schema_vuelos,
)

print(f"Total de filas: {df.count():,}")

[Stage 0:==========>                                              (3 + 13) / 16]

Total de filas: 7,079,081


In [3]:
# Solo 3 meses para que sea manejable
df_q1 = df.filter((col("month") >= 1) & (col("month") <= 3))
print(f"Filas Q1: {df_q1.count():,}")

[Stage 3:===>                                                     (1 + 15) / 16]

Filas Q1: 1,658,259


In [4]:
df_prep = (
    df_q1
    .withColumn("fecha", to_date(col("fl_date"), "yyyy-MM-dd"))
    .withColumn(
        "crs_dep_hour",
        (col("crs_dep_time").cast("int") / 100).cast("int")
    )
    .withColumn("dow", dayofweek(col("fecha")))
)

print("✅ Transformaciones listas")

✅ Transformaciones listas


In [5]:
# Agrupar: cuántos vuelos salen de cada aeropuerto, cada día, cada hora
df_agg = (
    df_prep
    .groupBy("origin", "fecha", "crs_dep_hour", "dow", "month")
    .agg(
        count("*").alias("vuelos_por_hora"),
        avg("dep_delay").alias("delay_promedio"),
        avg("distance").alias("distancia_promedio"),
    )
)

print(f"Registros agregados: {df_agg.count():,}")
df_agg.show(5)

Registros agregados: 249,148


[Stage 12:=============================================>          (13 + 3) / 16]

+------+----------+------------+---+-----+---------------+-------------------+------------------+
|origin|     fecha|crs_dep_hour|dow|month|vuelos_por_hora|     delay_promedio|distancia_promedio|
+------+----------+------------+---+-----+---------------+-------------------+------------------+
|   CLE|2024-01-01|          14|  2|    1|              8|              -0.75|           639.375|
|   ATL|2024-01-01|           9|  2|    1|             74|0.12162162162162163| 823.1216216216217|
|   TLH|2024-01-01|          11|  2|    1|              2|               -4.0|             312.5|
|   DHN|2024-01-01|           6|  2|    1|              1|               -6.0|             170.0|
|   EYW|2024-01-01|           7|  2|    1|              2|                0.0|             691.0|
+------+----------+------------+---+-----+---------------+-------------------+------------------+
only showing top 5 rows


In [6]:
# Calcular percentiles para clasificar congestión
percentiles = df_agg.approxQuantile("vuelos_por_hora", [0.33, 0.66], 0.01)
umbral_bajo = percentiles[0]
umbral_alto = percentiles[1]

print(f"Umbral bajo (P33): {umbral_bajo}")
print(f"Umbral alto (P66): {umbral_alto}")

# Crear variable objetivo
df_agg = df_agg.withColumn(
    "congestion_level",
    when(col("vuelos_por_hora") <= umbral_bajo, 0.0)
    .when(col("vuelos_por_hora") <= umbral_alto, 1.0)
    .otherwise(2.0)
)

# Ver distribución
df_agg.groupBy("congestion_level").count().orderBy("congestion_level").show()

Umbral bajo (P33): 1.0
Umbral alto (P66): 5.0


[Stage 21:=============================================>          (13 + 3) / 16]

+----------------+-----+
|congestion_level|count|
+----------------+-----+
|             0.0|90452|
|             1.0|84281|
|             2.0|74415|
+----------------+-----+



In [7]:
antes = df_agg.count()
df_clean = df_agg.dropDuplicates(["origin", "fecha", "crs_dep_hour"])
df_clean = df_clean.na.drop(subset=[
    "vuelos_por_hora", "delay_promedio",
    "distancia_promedio", "congestion_level"
])
despues = df_clean.count()

print(f"Antes: {antes:,}")
print(f"Después: {despues:,}")
print(f"Eliminadas: {antes - despues:,}")

[Stage 35:>                                                         (0 + 4) / 4]

Antes: 249,148
Después: 246,929
Eliminadas: 2,219


In [8]:
df_clean.write.mode("overwrite") \
    .partitionBy("month") \
    .parquet(f"{ARTIFACTS}/congestion_silver")
print("✅ Silver guardado")

✅ Silver guardado


In [20]:
# Leer de vuelta la salida particionada
df_verificacion = spark.read.parquet(f"{ARTIFACTS}/congestion_silver")

print(f"Filas leídas: {df_verificacion.count():,}")
print(f"Columnas: {df_verificacion.columns}")

# Verificar ida y vuelta
assert df_verificacion.count() == df_clean.count(), "¡Se perdieron filas!"
print("✅ Verificación de ida y vuelta OK")

Filas leídas: 246,929
Columnas: ['origin', 'fecha', 'crs_dep_hour', 'dow', 'vuelos_por_hora', 'delay_promedio', 'distancia_promedio', 'congestion_level', 'month']


[Stage 294:>                                                        (0 + 4) / 4]

✅ Verificación de ida y vuelta OK


In [21]:
# Verificar que Spark usa PartitionFilters al filtrar por month
df_verificacion.filter(col("month") == 1).explain(True)

== Parsed Logical Plan ==
'Filter '`=`('month, 1)
+- Relation [origin#1797,fecha#1798,crs_dep_hour#1799,dow#1800,vuelos_por_hora#1801L,delay_promedio#1802,distancia_promedio#1803,congestion_level#1804,month#1805] parquet

== Analyzed Logical Plan ==
origin: string, fecha: date, crs_dep_hour: int, dow: int, vuelos_por_hora: bigint, delay_promedio: double, distancia_promedio: double, congestion_level: double, month: int
Filter (month#1805 = 1)
+- Relation [origin#1797,fecha#1798,crs_dep_hour#1799,dow#1800,vuelos_por_hora#1801L,delay_promedio#1802,distancia_promedio#1803,congestion_level#1804,month#1805] parquet

== Optimized Logical Plan ==
Filter (isnotnull(month#1805) AND (month#1805 = 1))
+- Relation [origin#1797,fecha#1798,crs_dep_hour#1799,dow#1800,vuelos_por_hora#1801L,delay_promedio#1802,distancia_promedio#1803,congestion_level#1804,month#1805] parquet

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [origin#1797,fecha#1798,crs_dep_hour#1799,dow#1800,vuelos_por_hora#180

In [9]:
categoricas = ["origin"]
numericas = ["crs_dep_hour", "dow", "vuelos_por_hora",
             "delay_promedio", "distancia_promedio"]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="skip")
    for c in categoricas
]
encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_oh")
    for c in categoricas
]

predictores = numericas + [f"{c}_oh" for c in categoricas]
assembler = VectorAssembler(inputCols=predictores, outputCol="features")

df_ml = df_clean.select(numericas + categoricas + ["congestion_level"])

print("✅ Pipeline ML preparado")

✅ Pipeline ML preparado


In [10]:
pipeline_prep = Pipeline(stages=indexers + encoders + [assembler])
df_ml_prep = (
    pipeline_prep.fit(df_ml)
    .transform(df_ml)
    .select("features", "congestion_level")
)

df_ml_prep.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Total preparado: {df_ml_prep.count():,}")

df_train, df_test = df_ml_prep.randomSplit([0.8, 0.2], seed=42)
df_train.persist(StorageLevel.MEMORY_AND_DISK)
df_test.persist(StorageLevel.MEMORY_AND_DISK)

print(f"Entrenamiento: {df_train.count():,}")
print(f"Prueba: {df_test.count():,}")

Total preparado: 246,929


Entrenamiento: 197,788
Prueba: 49,141


In [14]:
def evaluar_clf(predicciones, nombre):
    resultados = {}

    # Métricas multiclase
    for metrica in ["accuracy", "f1", "weightedPrecision"]:
        evaluador = MulticlassClassificationEvaluator(
            labelCol="congestion_level",
            predictionCol="prediction",
            metricName=metrica
        )
        resultados[metrica.upper()] = evaluador.evaluate(predicciones)

    # AUC (binaria, necesita label 0/1)
    # Para multiclase usamos el weighted AUC manual
    print(f"{nombre}: Accuracy={resultados['ACCURACY']:.4f} F1={resultados['F1']:.4f}")
    return resultados

In [12]:
lr = LogisticRegression(
    featuresCol="features",
    labelCol="congestion_level",
    maxIter=20,
    regParam=0.01
)
modelo_lr = lr.fit(df_train)
pred_lr = modelo_lr.transform(df_test)

resultados_lr = evaluar_clf(pred_lr, "LogisticRegression")

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
                                                                                

LogisticRegression: Accuracy=0.8065 F1=0.8075


In [13]:
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="congestion_level",
    numTrees=20,
    maxDepth=5,
    minInstancesPerNode=5,
    seed=42
)
modelo_rf = rf.fit(df_train)
pred_rf = modelo_rf.transform(df_test)

resultados_rf = evaluar_clf(pred_rf, "Random Forest")

Random Forest: Accuracy=0.8400 F1=0.8393


In [15]:
importancias_array = modelo_rf.featureImportances.toArray()
print(f"Total features: {len(importancias_array)}")
print(f"Suma: {importancias_array.sum():.4f}")
print()

top_indices = importancias_array.argsort()[-10:][::-1]
print("Top 10 features por importancia:")
for rank, idx in enumerate(top_indices, 1):
    print(f"  {rank:2d}. Feature[{idx:3d}] = {importancias_array[idx]:.4f}")

Total features: 338
Suma: 1.0000

Top 10 features por importancia:
   1. Feature[  3] = 0.1706
   2. Feature[  2] = 0.1455
   3. Feature[  4] = 0.0962
   4. Feature[ 12] = 0.0708
   5. Feature[ 26] = 0.0561
   6. Feature[ 15] = 0.0504
   7. Feature[ 29] = 0.0317
   8. Feature[ 35] = 0.0315
   9. Feature[ 31] = 0.0265
  10. Feature[ 16] = 0.0234


In [16]:
import pandas as pd

comparacion = [
    {"Modelo": "LogisticRegression", **resultados_lr},
    {"Modelo": "Random Forest", **resultados_rf},
]

tabla = pd.DataFrame(comparacion)[["Modelo", "ACCURACY", "F1"]]
tabla = tabla.sort_values("F1", ascending=False).reset_index(drop=True)
tabla

,Modelo,ACCURACY,F1
0,Random Forest,0.839991,0.839251
1,LogisticRegression,0.806536,0.807505


In [17]:
df_comp = pd.DataFrame(comparacion)
ganador_idx = df_comp["F1"].idxmax()
ganador_nombre = df_comp.loc[ganador_idx, "Modelo"]
print(f"🏆 Ganador: {ganador_nombre}")
print(f"   F1 = {df_comp.loc[ganador_idx, 'F1']:.4f}")
print()

if ganador_nombre == "Random Forest":
    modelo_ganador = modelo_rf
else:
    modelo_ganador = modelo_lr

modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_congestion_ganador")
print(f"✅ Modelo guardado")

🏆 Ganador: Random Forest
   F1 = 0.8393



✅ Modelo guardado


## Fase 6 — Cierre

**Reflexión técnica:**

El modelo de congestión clasifica cada aeropuerto+hora en 3 niveles
(baja/media/alta). Random Forest tiende a superar a LogisticRegression
porque la congestión no depende linealmente de la hora y el día — hay
patrones complejos (mañana vs tarde, lunes vs domingo).

**Features más importantes:**
- `vuelos_por_hora`: predictor directo de congestión (por definición)
- `crs_dep_hour`: la hora del día captura el ciclo operativo
- `origin`: algunos aeropuertos son naturalmente más congestionados

**¿Por qué F1 y no solo accuracy?**
Porque las clases están desbalanceadas (más horas "bajas" que "altas").
Accuracy puede ser alta simplemente prediciendo siempre la clase mayoritaria.
F1 penaliza los errores en las clases minoritarias.

In [18]:
df_ml_prep.unpersist()
df_train.unpersist()
df_test.unpersist()
print("✅ Memoria liberada")

✅ Memoria liberada
